# PRUEBAS DE GESTIÓN DE VALORES NULOS

En este notebook se realizan pruebas manuales de imputación de valores nulos para validar qué estrategia resulta más adecuada en cada variable.

Estas pruebas permiten justificar las decisiones tomadas durante la fase de transformación y sirven como base para la posterior automatización mediante funciones reutilizables en el archivo `soporte_nulos.py`.

In [10]:
# Importación de librerías

import pandas as pd
import numpy as np

# Configuración para visualizar todas las columnas del DataFrame
pd.set_option("display.max_columns", None)

In [11]:
# Carga del dataset
df_hr = pd.read_csv("../../data/raw/hr.csv")

# Creamos una copia de trabajo para preservar el dataset original
df_hr_copia = df_hr.copy()

In [12]:
# Eliminación de columnas irrelevantes o redundantes

df_hr_copia.drop(["EmployeeCount", "StandardHours", "Over18"], axis=1, inplace=True)

df_hr_copia.drop(["DailyRate", "HourlyRate", "MonthlyRate"], axis=1, inplace=True)

df_hr_copia.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1357 non-null   str    
 3   Department                1445 non-null   str    
 4   DistanceFromHome          1474 non-null   int64  
 5   Education                 1474 non-null   int64  
 6   EducationField            1416 non-null   str    
 7   EmployeeNumber            1474 non-null   int64  
 8   EnvironmentSatisfaction   1474 non-null   int64  
 9   Gender                    1474 non-null   str    
 10  JobInvolvement            1474 non-null   int64  
 11  JobLevel                  1474 non-null   int64  
 12  JobRole                   1474 non-null   str    
 13  JobSatisfaction           1445 non-null   float64
 14  MaritalStatus      

In [13]:
# Eliminación de registros duplicados

df_hr_copia.drop_duplicates(inplace=True)

print(f"Dimensiones tras eliminar duplicados: {df_hr_copia.shape}")

Dimensiones tras eliminar duplicados: (1470, 29)


## Estandarización y corrección de variables categóricas

In [14]:
# Eliminamos espacios innecesarios y unificamos el estilo de escritura en "JobRole"

df_hr_copia["JobRole"] = df_hr_copia["JobRole"].str.strip().str.title()

In [15]:
# Corrección de error ortográfico detectado en "MaritalStatus"

df_hr_copia['MaritalStatus'] = df_hr_copia['MaritalStatus'].replace("Marreid", "Married")

# Revisión de que los cambios se hayan aplicado
df_hr_copia['MaritalStatus'].unique()

<StringArray>
['Single', 'Married', 'Divorced', nan]
Length: 4, dtype: str

## Revisión inicial de valores nulos

In [16]:
# Porcentaje de valores nulos por columna

porcentaje_nulos = round((df_hr_copia.isnull().sum() / df_hr_copia.shape[0] * 100), 2)

porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)

YearsWithCurrManager     10.00
MaritalStatus             8.98
BusinessTravel            7.96
TrainingTimesLastYear     5.99
Age                       4.97
EducationField            3.95
OverTime                  2.99
Department                1.97
JobSatisfaction           1.97
MonthlyIncome             0.95
dtype: float64

## Pruebas manuales de imputación

In [17]:
# YearsWithCurrManager → mediana por JobLevel

df_test = df_hr_copia.copy()

mediana_job_level = df_test.groupby("JobLevel")["YearsWithCurrManager"].transform("median")
df_test["YearsWithCurrManager"] = df_test["YearsWithCurrManager"].fillna(mediana_job_level)

print("Nulos restantes:", df_test["YearsWithCurrManager"].isnull().sum())

Nulos restantes: 0


In [18]:
# MaritalStatus → moda

moda = df_test["MaritalStatus"].mode()[0]
df_test["MaritalStatus"] = df_test["MaritalStatus"].fillna(moda)

print("Moda utilizada:", moda)
print("Nulos restantes:", df_test["MaritalStatus"].isnull().sum())

Moda utilizada: Married
Nulos restantes: 0


In [19]:
# BusinessTravel → moda

moda = df_test["BusinessTravel"].mode()[0]
df_test["BusinessTravel"] = df_test["BusinessTravel"].fillna(moda)

print("Moda utilizada:", moda)
print("Nulos restantes:", df_test["BusinessTravel"].isnull().sum())

Moda utilizada: Travel_Rarely
Nulos restantes: 0


In [20]:
# TrainingTimesLastYear → moda

moda = df_test["TrainingTimesLastYear"].mode()[0]
df_test["TrainingTimesLastYear"] = df_test["TrainingTimesLastYear"].fillna(moda)

print("Moda utilizada:", moda)
print("Nulos restantes:", df_test["TrainingTimesLastYear"].isnull().sum())

Moda utilizada: 2.0
Nulos restantes: 0


In [21]:
# Age → mediana por JobLevel

mediana_job_level_2 = df_test.groupby("JobLevel")["Age"].transform("median")
df_test["Age"] = df_test["Age"].fillna(mediana_job_level_2)

print("Nulos restantes:", df_test["Age"].isnull().sum())

Nulos restantes: 0


In [22]:
# EducationField → moda por Department + moda global

df_test["EducationField"] = df_test["EducationField"].str.strip().str.title()

df_test['EducationField'] = df_test.groupby('Department')['EducationField'].transform(lambda x: x.fillna(x.mode()[0]))

moda_global = df_test["EducationField"].mode()[0]
df_test["EducationField"] = df_test["EducationField"].fillna(moda_global)

print("Moda global utilizada:", moda_global)
print("Nulos restantes:", df_test["EducationField"].isnull().sum())

Moda global utilizada: Life Sciences
Nulos restantes: 0


In [23]:
# OverTime → valor fijo ("Unknown")

df_test["OverTime"] = df_test["OverTime"].fillna("Unknown")

print("Nulos restantes:", df_test["OverTime"].isnull().sum())

Nulos restantes: 0


In [24]:
# Department → moda por JobRole

df_test["Department"] = df_test.groupby("JobRole")["Department"].transform(lambda x: x.fillna(x.mode()[0]))

print("Nulos restantes:", df_test["Department"].isnull().sum())

Nulos restantes: 0


In [25]:
# MonthlyIncome → mediana por JobRole

df_test["MonthlyIncome"] = df_test.groupby("JobRole")["MonthlyIncome"].transform(lambda x: x.fillna(x.median()))

print("Nulos restantes:", df_test["MonthlyIncome"].isnull().sum())

Nulos restantes: 0


## Validación final conjunta

Una vez comprobada la lógica de imputación variable a variable, aplicamos todas las decisiones sobre una misma copia del dataset para verificar el resultado global.

## Conclusión

Las pruebas manuales realizadas permiten validar que las estrategias seleccionadas para la imputación de nulos son coherentes con la naturaleza de cada variable.

A partir de estas comprobaciones, las imputaciones más homogéneas se trasladan posteriormente al archivo `soporte_nulos.py`, mientras que las imputaciones más específicas se mantienen de forma explícita en la fase de transformación.